# Evaluating an encoder

This notebook evaluates one encoder on the registered tasks, excludes datasets whose images the encoder may have seen during pretraining, and inspects the leaderboard and the per image test results. Open datasets are streamed from their official hosts, so nothing has to be downloaded beforehand.

In [ ]:
import os
from pathlib import Path

import pandas as pd

from find_those_polyps import EvaluationConfig, configure_logging, evaluate, load_registry
from find_those_polyps.baselines import RandomResNet50
from find_those_polyps.data.__main__ import state
from find_those_polyps.evaluation.runner import select_tasks

configure_logging("INFO")
# Streamed files are cached under the data root, and local copies placed there are used directly.
DATA_ROOT = Path(os.environ.get("FIND_THOSE_POLYPS_DATA", "../data"))

## Registered datasets and tasks

The registry documents the licence, access route, source family and overlaps of every dataset. Leakage warnings and the partition sizes of every task are rendered in `docs/datasets.md`. The state column tells whether a dataset is local, partly cached, streamable or needs manual access.

In [ ]:
registry = load_registry()
cards = {card.key: card for card in registry.datasets}
datasets = pd.DataFrame(
    [
        card.model_dump(include={"key", "name", "access", "licence", "family"})
        | {"state": state(card, DATA_ROOT)}
        for card in registry.datasets
    ]
)
tasks = pd.DataFrame(
    [
        task.model_dump(include={"id", "dataset", "kind", "primary_metric"})
        | {"classes": len(task.classes), "state": state(cards[task.dataset], DATA_ROOT)}
        for task in registry.tasks
    ]
)
print(f"registry version {registry.version}")
display(datasets, tasks)

## Choosing the encoder

Any `Encoder` works. Replace the random baseline with your model, giving it a unique name and a one sentence description. `TorchEncoder` wraps a plain module with optional `preprocess`, `postprocess` and `postprocess_dense` callables.

In [ ]:
encoder = RandomResNet50(seed=0)
encoder.card

## Selecting and excluding tasks

`tasks` keeps identifiers matching glob patterns, and `exclude_tasks` and `exclude_datasets` remove further tasks. `pretrained_on` lists the datasets used to pretrain the encoder, and every task whose dataset is documented to overlap one of them is skipped. The example below runs two small tasks and shows which tasks a HyperKvasir pretrained encoder could be scored on. Use `tasks=("*",)` for the full benchmark.

In [ ]:
config = EvaluationConfig(
    tasks=("cls/hyperkvasir-bbps", "seg/kvasir-seg"),
    exclude_datasets=(),
    pretrained_on=(),
    data_root=DATA_ROOT,
    # The notebook writes apart from the benchmark leaderboard in results.
    leaderboard=Path("../results/notebook/leaderboard.csv"),
    per_image_dir=Path("../results/notebook/per_image"),
)
hyperkvasir_pretrained = config.model_copy(
    update={"tasks": ("*",), "pretrained_on": ("hyperkvasir-unlabeled",)}
)
print("kept for a HyperKvasir pretrained encoder")
print([spec.id for spec in select_tasks(registry, hyperkvasir_pretrained)])

In [ ]:
rows = evaluate(encoder, config)

## Leaderboard

Each row holds one model on one task, protocol and label budget, with one column per metric mean and one per standard deviation. `score` is the primary metric of the configuration with its 95% bootstrap interval over test images, and `rank` orders models within it. `leaderboard.xlsx` beside the CSV gives every configuration its own sheet.

In [ ]:
board = pd.read_csv(config.leaderboard)
board.pivot_table(index=["task", "protocol", "budget"], columns="model", values="score")

In [ ]:
board[board["task"] == "seg/kvasir-seg"][
    ["configuration", "model", "score", "score_ci_low", "score_ci_high", "dice", "nsd"]
]

## Per image errors

Each task stores one CSV per model with the prediction for every test image and its canonical path under the data root.

In [ ]:
task_id = "cls/hyperkvasir-bbps"
records = pd.read_csv(config.per_image_dir / task_id.replace("/", "__") / f"{encoder.name}.csv")
full = records[(records["protocol"] == "linear") & (records["budget"] == "full")]
confident_errors = full[~full["correct"]].sort_values("confidence", ascending=False)
confident_errors[["sample_id", "path", "target", "prediction", "confidence"]].head(20)